# NASA Li-ion Battery Prognostics: EDA & Baseline Modeling

This notebook provides a starter pipeline for analyzing the **NASA Prognostics Center of Excellence (PCoE) Li-ion Battery Aging Dataset**.

### Objectives
1. **Exploratory Data Analysis (EDA):** Inspect capacity degradation, cycle dynamics, and discharge profiles.
2. **Feature Engineering:** Extract cycle-level summaries (voltage drops, duration, temperature peaks).
3. **Prognostics / RUL Modeling:** Build a baseline machine learning model for **State of Health (SoH)** and **Remaining Useful Life (RUL)** estimation.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

from nasa_battery import BatteryDataLoader

sns.set_theme(style='whitegrid')
print('Environment initialized successfully!')

## 1. Load Data
Let's load the benchmark battery datasets: **B0005, B0006, B0007, B0018**.

In [ ]:
data_dir = '../5. Battery Data Set'
loader = BatteryDataLoader(data_root=data_dir)

target_batteries = ['B0005', 'B0006', 'B0007', 'B0018']
summaries = []

for b in target_batteries:
    df_b = loader.extract_cycle_summary(b)
    summaries.append(df_b)

all_df = pd.concat(summaries, ignore_index=True)
discharge_df = all_df[(all_df['type'] == 'discharge') & (all_df['capacity'].notna())].copy()
discharge_df['cycle_num'] = discharge_df.groupby('battery_id').cumcount() + 1
discharge_df.head()

## 2. Capacity Fade Visualization

In [ ]:
plt.figure(figsize=(10, 5), dpi=100)
for b in target_batteries:
    subset = discharge_df[discharge_df['battery_id'] == b]
    plt.plot(subset['cycle_num'], subset['capacity'], label=b, lw=2)

plt.axhline(1.4, color='red', linestyle='--', label='EOL Threshold (1.4 Ah)')
plt.title('Capacity Degradation Over Discharge Cycles', fontsize=13)
plt.xlabel('Discharge Cycle Count')
plt.ylabel('Capacity (Ah)')
plt.legend()
plt.show()

## 3. Voltage Discharge Profiles Across Aging Cycles
Let's inspect how the voltage curve shifts as Battery B0005 ages.

In [ ]:
sample_cycles = [2, 50, 100, 200, 300, 500]
plt.figure(figsize=(10, 5), dpi=100)

for c_idx in sample_cycles:
    try:
        ts_df = loader.extract_time_series('B0005', c_idx)
        if ts_df['type'].iloc[0] == 'discharge':
            plt.plot(ts_df['Time'], ts_df['Voltage_measured'], label=f'Cycle {c_idx}')
    except Exception:
        pass

plt.title('B0005 Discharge Voltage Curves over Battery Life', fontsize=13)
plt.xlabel('Discharge Time (seconds)')
plt.ylabel('Terminal Voltage (V)')
plt.legend()
plt.show()

## 4. Baseline Remaining Useful Life (RUL) Modeling
Train on batteries **B0005, B0006, B0007** and evaluate out-of-sample on **B0018**.

In [ ]:
def compute_rul(df_group, eol_thresh=1.4):
    eol_cycles = df_group[df_group['capacity'] <= eol_thresh]['cycle_num']
    eol_idx = eol_cycles.iloc[0] if len(eol_cycles) > 0 else df_group['cycle_num'].max()
    df_group['RUL'] = (eol_idx - df_group['cycle_num']).clip(lower=0)
    return df_group

df_rul = discharge_df.groupby('battery_id', group_keys=False).apply(compute_rul)

features = ['cycle_num', 'duration_sec', 'v_min', 'v_max', 'temp_max']
target = 'RUL'

train_df = df_rul[df_rul['battery_id'].isin(['B0005', 'B0006', 'B0007'])].dropna(subset=features + [target])
test_df = df_rul[df_rul['battery_id'] == 'B0018'].dropna(subset=features + [target])

model = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(train_df[features], train_df[target])

preds = model.predict(test_df[features])
rmse = np.sqrt(mean_squared_error(test_df[target], preds))
mae = mean_absolute_error(test_df[target], preds)
r2 = r2_score(test_df[target], preds)

print('Test Set (Battery B0018) Evaluation:')
print(f'  RMSE: {rmse:.2f} cycles')
print(f'  MAE:  {mae:.2f} cycles')
print(f'  R^2:  {r2:.3f}')

plt.figure(figsize=(10, 5), dpi=100)
plt.plot(test_df['cycle_num'], test_df[target], label='True RUL', color='black', lw=2)
plt.plot(test_df['cycle_num'], preds, label='Predicted RUL (XGBoost)', color='dodgerblue', linestyle='--', lw=2)
plt.title('Remaining Useful Life Prediction for Battery B0018', fontsize=13)
plt.xlabel('Cycle Number')
plt.ylabel('RUL (Cycles remaining)')
plt.legend()
plt.show()